# One iteration of the loop

Every forecast you have run in this course held land use still. The scheme
changed costs, accessibility responded, and not one job moved in consequence.
This notebook lets them move.

You wrote the rule and chose its parameter. What follows is one iteration of the
loop: allocate the growth, re-run the distribution, re-compute accessibility,
then set the answer beside the one you would have had with land use fixed.

The run will look impeccable: balancing converges, all 290 zone margins match,
and the accessibility surface is as respectable as the one you mapped earlier.
Every check this course has taught you returns a clean verdict. And none of them
can tell you that you are one step into an unfinished sequence.

So the thing to carry away is the log at the bottom, which
records what went into the run beside what came out. A fixed-land-use forecast
needed one qualifier before anybody could read it, the scheme definition. This
one needs three more, and none is recoverable from the output.

Work through the sections in order, from the top of the page to the bottom.

## Where the files are

JupyterLite runs inside your browser. Nothing is installed on your machine and
you do not need administrator rights, which is why this page opens on a
locked-down work machine.

The notebook and its data arrived with the site, so there is nothing to download
and nothing to upload. Open the file browser - the panel down the left-hand
side, or the folder icon in the far-left sidebar if it is not showing - and you
will find this arrangement already in place:

```
running-the-loop/
    running-the-loop.ipynb
    data/
        zones_msoa.csv
        trip_ends_msoa.csv
        cost_matrix_msoa.csv
        scheme_cost_adjustments_narrow.csv
```

Every path in the code below assumes it. The notebook sits at the top of the
folder and the data sits one level under it, so moving either one will break the
loading section.

**What happens to anything you change**

Because there is no server behind this, whatever you save goes into your
browser's own storage rather than onto a network drive. That has two
consequences worth taking seriously. Anything you want to keep should be
downloaded - right-click the file in the file browser and choose **Download**.
And if you clear your browsing data, or if your employer's IT policy clears it
for you, your saved work goes with it.

Do not edit the CSV files. If you want to try something out on them, duplicate
one first and work on the copy.

**Getting back to the original**

Should you change the notebook and want the version you started with, use
**Help > Clear Browser Data**. But read the warning it gives you before
confirming. It removes everything you have stored on this site, for every
notebook here, and it cannot be undone, so download anything you care about
first.

## Checking the files are where you think they are

Run the cell below first. It reports what it can see, which is faster than
reading an error message later and guessing at the cause.

In [ ]:
import os

DATA_FOLDER = "data"

expected = [
    "zones_msoa.csv",
    "trip_ends_msoa.csv",
    "cost_matrix_msoa.csv",
    "scheme_cost_adjustments_narrow.csv",
]

print("Looking in:", os.path.abspath(DATA_FOLDER))
print()

if not os.path.isdir(DATA_FOLDER):
    print("That folder does not exist yet.")
    print("Check the folder names and check where this notebook is saved.")
else:
    found = sorted(os.listdir(DATA_FOLDER))
    for name in expected:
        status = "found" if name in found else "MISSING"
        print(f"  {name:38s} {status}")

## Parameters

One value, and the only thing in this notebook you will change.
ACCESS_ELASTICITY is the parameter you met on the page about writing the
reallocation rule, where it appeared as the Greek letter epsilon. The name here
and the symbol there are one quantity, and it carries no units.

Everything else is fixed and sits in the loading cell below. Beta stays at
0.1185 per minute, the opportunity variable at jobs, the scheme definition at
narrow, the growth to be allocated at 14,863 jobs. Were any of those free to
move here, you could not attribute a difference between two runs to the
elasticity.

Run the notebook at 1.0 first. Then set a value of your own choosing and run it
again, so the log has two rows in it.

In [ ]:
# ---------------------------------------------------------------------------
# PARAMETERS
# ---------------------------------------------------------------------------

ACCESS_ELASTICITY = 1.0    # how strongly jobs follow accessibility.
                           # 0.0 shares growth out in proportion to existing jobs and ignores the scheme entirely.

# ---------------------------------------------------------------------------

## Loading the data, and the scheme

Three data files and the scheme file. The cost matrix is much the largest at
21,025 rows, so give the cell a moment before deciding it has stalled.

The narrow scheme file is applied exactly as in the scheme test, 42 rows of the
matrix rewritten and nothing else touched. Which pairs a Metro extension
improves is settled, and it is not what this exercise varies.

In [ ]:
import numpy as np
import pandas as pd

BETA = 0.1185                    # fixed, per minute of generalised cost
OPPORTUNITY = "jobs"             # fixed
SCHEME_DEFINITION = "narrow"     # fixed
GROWTH_TOTAL = 14863             # jobs to be allocated, fixed
TOLERANCE = 0.01                 # balancing tolerance, in trips
MAX_ITERATIONS = 100

zones = pd.read_csv(f"{DATA_FOLDER}/zones_msoa.csv", encoding="utf-8-sig")
trip_ends = pd.read_csv(f"{DATA_FOLDER}/trip_ends_msoa.csv",
                        encoding="utf-8-sig")
costs = pd.read_csv(f"{DATA_FOLDER}/cost_matrix_msoa.csv",
                    encoding="utf-8-sig")

zone_ids = list(zones["zone_id"])
position = {zone: i for i, zone in enumerate(zone_ids)}

details = (zones[["zone_id", "zone_name", "local_authority"]]
           .merge(trip_ends[["zone_id", "resident_workers", "jobs"]],
                  on="zone_id")
           .set_index("zone_id")
           .reindex(zone_ids))

zone_names = list(details["zone_name"])
base_jobs = details[OPPORTUNITY].values.astype(float)
base_workers = details["resident_workers"].values.astype(float)

base_cost = (costs
             .pivot(index="origin_id", columns="destination_id",
                    values="gc_min")
             .reindex(index=zone_ids, columns=zone_ids)
             .values.astype(float))

adjustments = pd.read_csv(
    f"{DATA_FOLDER}/scheme_cost_adjustments_{SCHEME_DEFINITION}.csv",
    encoding="utf-8-sig", comment="#")

scheme_cost = base_cost.copy()
for origin, destination, change in adjustments.itertuples(index=False):
    i, j = position[origin], position[destination]
    scheme_cost[i, j] = base_cost[i, j] + change

print(f"Zones:                  {len(zone_ids)}")
print(f"Cost matrix:            {base_cost.shape[0]} by {base_cost.shape[1]}"
      f"   ({base_cost.size:,} ordered pairs)")
print(f"Resident workers:       {base_workers.sum():,.0f}")
print(f"Jobs:                   {base_jobs.sum():,.0f}")
print(f"Scheme rows applied:    {len(adjustments)}"
      f"   ({SCHEME_DEFINITION} definition)")
print(f"Beta:                   {BETA} per minute")
print(f"Growth to allocate:     {GROWTH_TOTAL:,} jobs")
print(f"ACCESS_ELASTICITY:      {ACCESS_ELASTICITY}")

## Where this run starts

The forecast below is the one you already have: the scheme written into the cost
matrix, and land use held still. Ten of the 145 zones gain any accessibility
and the other 135 gain precisely nothing,
because a zone whose row of the matrix did not change cannot move.

That proportional gain is the input to your rule. It is what a zone's share of
the growth is multiplied up by.

In [ ]:
base_access = np.exp(-BETA * base_cost) @ base_jobs
scheme_access = np.exp(-BETA * scheme_cost) @ base_jobs
gain = (scheme_access - base_access) / base_access

baseline = pd.DataFrame({
    "zone_id": zone_ids,
    "zone_name": zone_names,
    "jobs": base_jobs,
    "gain_pct": 100.0 * gain,
})

print("THE FIXED-LAND-USE FORECAST THIS RUN STARTS FROM")
print("-" * 70)
print(f"Zones gaining any accessibility: "
      f"{int((gain > 1e-9).sum())} of {len(zone_ids)}")
print()
print(baseline.nlargest(10, "gain_pct")
      .round({"jobs": 0, "gain_pct": 2}).to_string(index=False))

## Applying your rule

The growth now arrives. The 14,863 jobs are shared across the 145 zones by your
rule, in which a zone's weight is its base-year jobs multiplied by one plus your
elasticity times its proportional accessibility gain.

Because the same 14,863 jobs go somewhere whatever the elasticity, the cell
below reports your run against a flat allocation at zero rather than against
nothing at all. That is the only honest way to see what the elasticity did. A zone
in the lower half of the table has not lost jobs. It has received fewer of
the new ones than a share-of-existing-jobs rule would have given it.

In [ ]:
def allocate(elasticity):
    weight = base_jobs * (1.0 + elasticity * gain)
    return GROWTH_TOTAL * weight / weight.sum()


added_run = allocate(ACCESS_ELASTICITY)
added_ref = allocate(0.0)

jobs_run = base_jobs + added_run
jobs_ref = base_jobs + added_ref
moved = added_run - added_ref
moved_total = float(np.clip(moved, 0.0, None).sum())

allocation = pd.DataFrame({
    "zone_id": zone_ids,
    "zone_name": zone_names,
    "base_jobs": base_jobs,
    "added": added_run,
    "revised_jobs": jobs_run,
    "vs_flat": moved,
})

print("WHERE THE 14,863 JOBS WENT")
print("-" * 78)
print(f"Allocated at an elasticity of {ACCESS_ELASTICITY},"
      f" against a flat allocation at 0.0")
print(f"Jobs moved by the elasticity:   {moved_total:,.1f}"
      f"   ({100.0 * moved_total / GROWTH_TOTAL:.2f} per cent of the growth)")
print(f"Zones receiving more than flat: {int((moved > 1e-9).sum())}")
print(f"Zones receiving less than flat: {int((moved < -1e-9).sum())}")
print()

if moved_total > 1e-6:
    print("Largest gains and largest losses against a flat allocation")
    show = pd.concat([allocation.nlargest(6, "vs_flat"),
                      allocation.nsmallest(4, "vs_flat")])
    print(show.round({"base_jobs": 0, "added": 1, "revised_jobs": 1,
                      "vs_flat": 1}).to_string(index=False))
else:
    print("At an elasticity of 0.0 your run and the flat allocation are the")
    print("same run, so there is nothing to compare. That is the correct")
    print("answer and it is worth seeing once.")

## Re-running the distribution

Same balancing code, same tolerance of 0.01 trips, same 145 zones. Although
none of that changed, one thing had to be settled before the model would run.
The study area's 371,585 jobs matched its 371,585 resident workers exactly, and
putting 14,863 more on one side breaks the match a doubly constrained model
depends on.

Resident workers are therefore grown in the same proportion, a factor of 1.0400,
and shared out as people already live. It asserts nothing about anybody moving house, and it
holds the origin margins identical across your run and the flat one, so whatever
separates the two matrices came from the destination side where the elasticity
acted.

In [ ]:
growth_factor = jobs_run.sum() / base_workers.sum()
workers_run = base_workers * growth_factor


def balance(cost, origins, destinations):
    deterrence = np.exp(-BETA * cost)
    row_factor = np.ones(len(origins))
    col_factor = np.ones(len(destinations))
    for iteration in range(1, MAX_ITERATIONS + 1):
        row_factor = 1.0 / (deterrence * (col_factor * destinations)).sum(axis=1)
        col_factor = 1.0 / (deterrence
                            * (row_factor * origins)[:, None]).sum(axis=0)
        modelled = ((row_factor * origins)[:, None]
                    * (col_factor * destinations)[None, :]
                    * deterrence)
        row_error = np.abs(modelled.sum(axis=1) - origins).max()
        col_error = np.abs(modelled.sum(axis=0) - destinations).max()
        if max(row_error, col_error) < TOLERANCE:
            break
    return modelled, iteration, row_error, col_error


matrix_run, iters_run, row_err_run, col_err_run = balance(
    scheme_cost, workers_run, jobs_run)
matrix_ref, iters_ref, row_err_ref, col_err_ref = balance(
    scheme_cost, workers_run, jobs_ref)

mean_cost_run = (matrix_run * scheme_cost).sum() / matrix_run.sum()
mean_cost_ref = (matrix_ref * scheme_cost).sum() / matrix_ref.sum()

outside = int((np.abs(matrix_run.sum(axis=1) - workers_run) >= TOLERANCE).sum()
              + (np.abs(matrix_run.sum(axis=0) - jobs_run) >= TOLERANCE).sum())

print("CONVERGENCE AND MARGINS")
print("-" * 70)
print(f"Resident workers grown by:      {growth_factor:.4f}"
      f"   to {workers_run.sum():,.0f}")
print(f"Jobs after allocation:          {jobs_run.sum():,.0f}")
print(f"Margins agree:                  "
      f"{abs(workers_run.sum() - jobs_run.sum()) < 0.5}")
print()
print(f"Your run        iterations {iters_run:3d}"
      f"   worst row error {row_err_run:.4f} trips"
      f"   worst column error {col_err_run:.4f}")
print(f"Flat reference  iterations {iters_ref:3d}"
      f"   worst row error {row_err_ref:.4f} trips"
      f"   worst column error {col_err_ref:.4f}")
print()
print(f"Zone margins outside tolerance: {outside} of {2 * len(zone_ids)}")
print(f"Trips distributed:              {matrix_run.sum():,.0f}")
print()
print(f"Mean generalised cost of a modelled trip: {mean_cost_run:.4f} minutes")
print(f"  the same figure on the flat allocation: {mean_cost_ref:.4f} minutes")

## What the response did to accessibility

Two comparisons, and the difference between them matters more than either alone.

The first sets your run against the flat allocation. Both hold 386,448 jobs, so
the levels are comparable and what is left is redistribution. Expect small
movement: at an elasticity of 1.0 the rule shifts 3.34 per cent of the growth,
and once that has gone through the deterrence function no zone in Tyne and Wear
moves by half a per cent.

The second states the scheme's benefit as a percentage of each zone's
do-minimum accessibility, which cancels the growth out of both sides.
Read the count before the sizes. Under fixed land use exactly 10 zones showed
any benefit. Look at how many show one now, and at whether any has had one
minute taken off one journey.

In [ ]:
access_run = np.exp(-BETA * scheme_cost) @ jobs_run
access_ref = np.exp(-BETA * scheme_cost) @ jobs_ref
access_dominimum = np.exp(-BETA * base_cost) @ jobs_ref

surface = pd.DataFrame({
    "zone_id": zone_ids,
    "zone_name": zone_names,
    "local_authority": list(details["local_authority"]),
    "access_flat": access_ref,
    "access_run": access_run,
    "change_pct": 100.0 * (access_run - access_ref) / access_ref,
    "benefit_fixed_pct": 100.0 * (scheme_access - base_access) / base_access,
    "benefit_run_pct": (100.0 * (access_run - access_dominimum)
                        / access_dominimum),
})
surface["benefit_shift_pp"] = (surface["benefit_run_pct"]
                               - surface["benefit_fixed_pct"])

rising = int((surface["change_pct"] > 1e-9).sum())
falling = int((surface["change_pct"] < -1e-9).sum())

print("YOUR RUN AGAINST THE FLAT ALLOCATION")
print("-" * 78)
print(f"Zones rising:  {rising}")
print(f"Zones falling: {falling}")
print(f"Largest rise {surface['change_pct'].max():+.3f} per cent"
      f"   largest fall {surface['change_pct'].min():+.3f} per cent")
print()
if rising or falling:
    movers = pd.concat([surface.nlargest(6, "change_pct"),
                        surface.nsmallest(4, "change_pct")])
    print(movers[["zone_id", "zone_name", "change_pct"]]
          .round({"change_pct": 3}).to_string(index=False))
else:
    print("Nothing moved, because the elasticity is 0.0.")

print()
print("THE SCHEME'S BENEFIT: FIXED LAND USE, THEN THIS RUN")
print("-" * 78)
print(f"Zones showing any benefit under fixed land use: "
      f"{int((surface['benefit_fixed_pct'] > 1e-4).sum())}")
print(f"Zones showing any benefit under this run:       "
      f"{int((surface['benefit_run_pct'] > 1e-4).sum())}")
print()
print(surface.nlargest(10, "benefit_fixed_pct")
      [["zone_id", "zone_name", "benefit_fixed_pct", "benefit_run_pct",
        "benefit_shift_pp"]]
      .round(3).to_string(index=False))
print()
print("Largest benefits among zones the scheme file never named")
named = set(adjustments["origin_id"]) | set(adjustments["destination_id"])
unnamed = surface[~surface["zone_id"].isin(named)]
print(unnamed.nlargest(5, "benefit_run_pct")
      [["zone_id", "zone_name", "benefit_run_pct"]]
      .round(3).to_string(index=False))

## The results log

This is the deliverable. Not any single figure above it.

The cell below writes one row per run into `luti-run-log.csv`, keyed on the
elasticity, so running the same value twice replaces its row. Beside the outputs
it carries the inputs: the elasticity you set, the loop iterations run, the
growth assumed, the scheme file applied, and the beta and opportunity column
fixed for you.


Two different things there are called iterations, so the log names both.
`balancing_iterations` counts the scalings Furness needed before the margins
matched, a property of the arithmetic. `loop_iterations` counts the times land
use and transport have been round the loop, which is a decision somebody made.

Consider what the run would be worth without it. Somebody hands you an
accessibility surface for Washington under the Metro extension. It is well
formed and its numbers are plausible. You cannot tell whether the modeller let
land use respond, and if they did, at what elasticity and after how many
iterations they stopped. None of that is in the numbers and all of it changes
them. A run with no record of its parameters is not evidence. It is a figure
with a story nobody can check.

Download it before you leave the page.

In [ ]:
LOG = "luti-run-log.csv"

entry = {
    "run": f"elasticity_{ACCESS_ELASTICITY}",
    "access_elasticity": ACCESS_ELASTICITY,
    "loop_iterations": 1,
    "growth_total": GROWTH_TOTAL,
    "beta": BETA,
    "opportunity": OPPORTUNITY,
    "scheme_definition": SCHEME_DEFINITION,
    "worker_growth_factor": round(float(growth_factor), 4),
    "balancing_iterations": iters_run,
    "worst_row_error_trips": round(float(row_err_run), 4),
    "mean_trip_cost_min": round(float(mean_cost_run), 4),
    "jobs_moved_vs_flat": round(moved_total, 1),
    "zones_rising": rising,
    "zones_falling": falling,
    "largest_rise_pct": round(float(surface["change_pct"].max()), 3),
    "largest_fall_pct": round(float(surface["change_pct"].min()), 3),
    "zones_with_scheme_benefit": int((surface["benefit_run_pct"] > 1e-4).sum()),
    "benefit_shift_pp_washington": round(float(
        surface.loc[surface["zone_id"] == "E02001809",
                    "benefit_shift_pp"].iloc[0]), 3),
}

log = pd.DataFrame([entry])
if os.path.exists(LOG):
    previous = pd.read_csv(LOG, encoding="utf-8-sig")
    previous = previous[previous["run"] != entry["run"]]
    log = pd.concat([previous, log], ignore_index=True)

log = log.sort_values("access_elasticity").reset_index(drop=True)
log.to_csv(LOG, index=False, encoding="utf-8-sig")

surface.round(4).to_csv("accessibility-by-zone.csv", index=False,
                        encoding="utf-8-sig")

print("RESULTS LOG")
print("-" * 78)
for column in log.columns:
    values = "   ".join(str(value) for value in log[column])
    print(f"  {column:28s} {values}")
print()
print(f"Runs recorded: {len(log)}")
print()
print("Written: luti-run-log.csv")
print("Written: accessibility-by-zone.csv")
print("Right-click each one in the file browser and choose Download.")

## What this run cannot tell you

You have one iteration. It gives you the direction of the feedback and roughly
its size at the elasticity you chose, which is what you needed in order to judge
whether running the loop was worth the trouble. It tells you nothing about where
the sequence ends.

And there is a limit on even that. A single iteration can point the opposite way
from where the sequence settles, so direction of travel is a weaker signal than
it feels while you read it off a table this tidy. Whether it happens here is a
question for the runs that come next.

Which leaves the question this page cannot answer, and it is the one to write
down. Is this the answer, or where we stopped?